In [ ]:
!git clone --depth 1 -b main https://github.com/DavutKursun/sql-tuner /tmp/sql-tuner
%cd /tmp/sql-tuner
!git log -1 --format='commit %h %s'
!pip install -q -r requirements-train.txt && pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
!python scripts/download.py --fetch hf && python scripts/build_splits.py > /tmp/splits.log && tail -5 /tmp/splits.log

In [ ]:
!python scripts/train_grpo.py --base Qwen/Qwen2.5-Coder-1.5B-Instruct --sft-adapter "/kaggle/input/**/models/sft" --steps 5 --group-size 8 --prompts-per-step 4 --beta 0.02 --lr 5e-6 --exec-bonus 0.1 --settings-from eval/results/grpo_probe.json --eval-every 1000 --val-subset eval/results/val_subset_ids.json --out models/grpo-timing --tag timing && echo "1 ok" >> /tmp/cmd_status || echo "1 FAILED" >> /tmp/cmd_status

In [ ]:
!mkdir -p /kaggle/working/out && cp -r models eval /kaggle/working/out/ && du -sh /kaggle/working/out/*

In [ ]:
import json, pathlib
cmds = ["python scripts/train_grpo.py --base Qwen/Qwen2.5-Coder-1.5B-Instruct --sft-adapter \"/kaggle/input/**/models/sft\" --steps 5 --group-size 8 --prompts-per-step 4 --beta 0.02 --lr 5e-6 --exec-bonus 0.1 --settings-from eval/results/grpo_probe.json --eval-every 1000 --val-subset eval/results/val_subset_ids.json --out models/grpo-timing --tag timing"]
done = dict(line.split(' ', 1) for line in pathlib.Path('/tmp/cmd_status').read_text().split('\n') if line) if pathlib.Path('/tmp/cmd_status').exists() else {}
status = [{'cmd': c, 'status': done.get(str(i), 'not run')} for i, c in enumerate(cmds, 1)]
pathlib.Path('/kaggle/working/out').mkdir(parents=True, exist_ok=True)
pathlib.Path('/kaggle/working/out/status.json').write_text(json.dumps(status, indent=1))
print('COMMANDS:', [s['status'] for s in status])